<h2>Załadowanie dataseta i odczytanie podstawowych informacji</h2>


In [1]:
import pandas as pd

df = pd.read_csv("./../data/corrected_phishing_dataset.csv")

print("Liczba rekordów:", df.shape[0])
print("Liczba kolumn:", df.shape[1])

print("\nInformacje o kolumnach:")
df.info()
TARGET = "type"

df[TARGET].value_counts()


Liczba rekordów: 651191
Liczba kolumn: 2

Informacje o kolumnach:
<class 'pandas.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   url     651191 non-null  str  
 1   type    651119 non-null  str  
dtypes: str(2)
memory usage: 9.9 MB


type
benign        428209
defacement     96457
phishing       93933
malware        32520
Name: count, dtype: int64

<h2>Odszukanie brakujących wartości w każdej kolumnie i usunięcie rekordów z nimi powiązanych</h2>


In [2]:
print("\nLiczba brakujących wartości w każdej kolumnie:")
missing = df.isna().sum()
missing = missing[missing>0]
print(missing.to_string())
print("Przed:", len(df))

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("Po:", len(df))
print("Brakujące etykiety:", df[TARGET].isna().sum())


Liczba brakujących wartości w każdej kolumnie:
type    72
Przed: 651191
Po: 651119
Brakujące etykiety: 0


<h2>Sprawdzenie ilości duplikatów URL oraz ich usunięcie</h2>

In [3]:
print("Duplikaty URL:", df.duplicated(subset=["url"]).sum())
conflicting_labels = (
    df.groupby("url")[TARGET]
    .nunique()
)

print(
    "URL z różnymi etykietami:",
    (conflicting_labels > 1).sum()
)
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print("Usunięte duplikaty:", before - len(df))
print("Pozostałe rekordy:", len(df))

Duplikaty URL: 10072
URL z różnymi etykietami: 0
Usunięte duplikaty: 10072
Pozostałe rekordy: 641047


<h2>Wykrycie rekordów zaczynających i kończących się od symbolu ' oraz ich oczyszczenie</h2>

In [4]:
#Czyszczenie rekordów zaczynających się znakiem apostrofu, nie były poprawnie parsowane przez spliturl
# Identyfikacja rekordów do poprawienia
mask = df["url"].str.startswith("'") & \
       df["url"].str.endswith("'")

print("Liczba URL-i wymagających poprawienia:", mask.sum())
display(df.loc[mask, ["url", "type"]].head(20))
df["url"] = df["url"].str.strip("'")

Liczba URL-i wymagających poprawienia: 3900


,url,type
545195,'motors.shop.ebay.com-cars-trucks-724527.jnq3....,phishing
545209,'www.gestion-des-impayes.com/visuel.php?param=...,phishing
545286,'nicolecustodio.com.br/paypal\%20us/webscr.htm...,phishing
545372,'beforenanny911.com/auto/my-themes/file/proper...,phishing
545379,'www.edyshsdf32.hut4.ru/Redirecionamento.html?...,phishing
545419,'www.google.ad/url?sa=f&rct=j&url=https:/www.p...,phishing
545503,'justoneofthosethings.co.uk/bahi\%20paypal/sec...,phishing
545509,'justoneofthosethings.co.uk/bahi\%20paypal/sec...,phishing
545551,'justoneofthosethings.co.uk/bahi\%20paypal/sec...,phishing
545552,'justoneofthosethings.co.uk/bahi\%20paypal/',phishing


<h2>Wykrycie rekordów, które nie mogą zostać sparsowane przez urlsplit oraz ich usunięcie</h2>

In [5]:
import re
from urllib.parse import urlsplit
from IPython.display import display

SCHEME_PATTERN = re.compile(r"^[A-Za-z][A-Za-z0-9+.-]*://")
CONTROL_PATTERN = re.compile(r"[\x00-\x1F\x7F-\x9F]")

VALID_LABELS = {
    "benign",
    "phishing",
    "malware",
    "spam",
    "defacement"
}


def is_valid_url(url):
    if not isinstance(url, str) or not url.strip():
        return False

    if CONTROL_PATTERN.search(url):
        return False

    try:
        if SCHEME_PATTERN.match(url) or url.startswith("//"):
            parsed = urlsplit(url)
        else:
            parsed = urlsplit("//" + url)
        return bool(parsed.hostname)

    except ValueError:
        return False


# 1. Walidacja adresów URL
valid_url_mask = df["url"].apply(is_valid_url)

valid_mask = valid_url_mask

# 4. Wyodrębnienie odrzuconych rekordów
df_invalid = df.loc[~valid_mask].copy()

print("Liczba wszystkich rekordów:", len(df))
print("Nieprawidłowe URL-e:", (~valid_url_mask).sum())
print("Łącznie odrzuconych rekordów:", len(df_invalid))

display(df_invalid[["url", "type"]])

# 5. Usunięcie nieprawidłowych rekordów
df = df.loc[valid_mask].copy().reset_index(drop=True)

print("Rekordy po filtrowaniu:", len(df))

Liczba wszystkich rekordów: 641047
Nieprawidłowe URL-e: 34
Łącznie odrzuconych rekordów: 34


,url,type
61370,http://www.scsrxx.com/js/?https://secure.runes...,phishing
520493,http://gkjx168.com/images?us.battle.net/login/...,phishing
523151,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523240,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523243,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523244,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523245,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523255,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523256,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523259,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing


Rekordy po filtrowaniu: 641013


<h2>Zapisanie oczyszczonego dataseta do osobnego pliku CSV</h2>
Na podstawie tego pliku wykonana zostanie ekstrakcja cech.

In [6]:
df.to_csv('./../data/deduplicated_corrected_phishing_dataset.csv',index=False, encoding="utf-8")

<h2>Wywołanie funkcji ekstrakcji cech, która zapisuje wzbogacony dataset do pliku urls_features.csv</h2>

In [7]:
from src.features import main
main()

Wczytano 641013 rekordów.
Wyekstraktowano 30 cech.
Zapisano dane do: ..\data\urls_features.csv

Brakujące wartości w nowych cechach:
domain                      0
tld                         0
suffix                      0
subdomain                   0
url_length                  0
domain_length               0
subdomain_length            0
suffix_length               0
host_length                 0
path_length                 0
arguments_url_ratio         0
path_url_ratio              0
arguments_domain_ratio      0
arguments_hostname_ratio    0
subdomain_domain_ratio      0
suffix_tld_ratio            0
longest_url_token_length    0
average_url_token_length    0
dot_count                   0
hyphen_count                0
underscore_count            0
question_mark_count         0
equal_sign_count            0
ampersand_count             0
has_https                   0
domain_entropy              0
subdomain_entropy           0
symbol_count                0
digit_count                

<h2>Zakończenie pracy wstępnego notebooka</h2>